# Visualizing Circuit Representation

{py:mod}`claveles.circuit.visualize` draws a circuit representation, which is the quickest way to check that
a circuit holds what you meant it to — in particular where its feedforward runs from and to.

* {py:func}`~claveles.circuit.visualize.make_figure` returns a {py:class}`matplotlib.figure.Figure`, which
  Jupyter displays and {py:func}`matplotlib.pyplot.show` shows.
* {py:func}`~claveles.circuit.visualize.savefig` writes one straight to a file.

A state placed on a mode is drawn as a box at the left end of its line, labelled
{py:data}`~claveles.circuit.visualize.DEFAULT_STATE_SYMBOL`; a mode left **open** has none, which is how
a figure says which is which. Every state is drawn with that one symbol, so a figure says *which*
modes carry a state rather than which state each one carries. The sample circuit below places none,
leaving every mode open.

Both take the same keyword options, listed in {ref}`sec:viz-circuit-options`.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## A sample circuit

In [ ]:
from claveles.circuit import CircuitRepr

c = CircuitRepr("sample_circuit")

In [ ]:
from math import pi

from claveles.circuit.ops import std
from claveles.feedforward import feedforward

c.Q(0) | std.PhaseRotation(pi)
c.Q(1) | std.QGate(t=0.2)
c.Q(0, 2) | std.BeamSplitter(theta=1, phi=0.1)
c.Q(3) | std.QGate(t=0.2)


@feedforward
def f(x: float) -> float:
    return x + 1


x = c.Q(0) | std.MeasureHomodyne(phi=pi / 2)
y = c.Q(1) | std.MeasureHomodyne(phi=-pi / 2)
c.Q(2, 3) | std.TwoModeShear(a=f(x), b=f(y))

c.Q(2) | std.MeasureHomodyne(phi=pi / 2)
c.Q(3) | std.MeasureHomodyne(phi=-pi / 2)

Drawing it:

In [ ]:
from claveles.circuit.visualize import make_figure

make_figure(c);

A parameter that comes from a measurement has no value yet, so it is drawn as an arrow from the
measurement that supplies it, labelled with the parameter it drives.

{py:func}`~claveles.circuit.visualize.savefig` writes the same figure to a file. The format follows the
suffix:

In [ ]:
from claveles.circuit.visualize import savefig

savefig(c, "claveles_circuit_example.png")

(sec:viz-circuit-options)=

## Options

| Option | Default | Description |
| --- | --- | --- |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.show_parameters` | `True` | Whether each operation's parameter values are written beside its box. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.parameters_fontsize` | `8.0` | Font size of those values. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.show_feedforward_param_label` | `True` | Whether each feedforward arrow is labelled with the parameter it drives. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.feedforward_param_fontsize` | `8.0` | Font size of that label. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.feedforward_arrow_style` | `->` | Matplotlib arrow style of the feedforward arrows. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.feedforward_arrow_color` | `lightgreen` | Their colour, in any format matplotlib accepts. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.qumode_labels` | $q_i$ per mode | Labels of the mode lines, as a list of strings. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.show_states` | `True` | Whether a state placed on a mode is drawn as a box at the left end of its line. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.state_symbol` | {py:data}`~claveles.circuit.visualize.DEFAULT_STATE_SYMBOL` (`$\|\psi\rangle$`) | Symbol written in that box. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.state_color` | {py:data}`~claveles.circuit.visualize.DEFAULT_STATE_COLOR` (`white`) | Its colour. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.qumode_hline_spacing` | {py:data}`~claveles.circuit.visualize.DEFAULT_QUMODE_HLINE_SPACING` (`1.0`) | Vertical spacing between the mode lines. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.opbox_width` | {py:data}`~claveles.circuit.visualize.DEFAULT_OPBOX_WIDTH` (`1.0`) | Width of an operation box. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.opbox_vertical_margin` | {py:data}`~claveles.circuit.visualize.DEFAULT_OPBOX_VERTICAL_MARGIN` (`0.2`) | Vertical margin between operation boxes. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.show_op_legend` | `False` | Whether a legend of the operations the circuit uses is drawn in the upper right corner. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.user_operation_name_table` | {py:data}`~claveles.circuit.visualize.DEFAULT_USER_OP_NAME_TABLE` | Symbol written in the box of each {py:mod}`~claveles.circuit.ops.std` operation, as a dictionary from operation name to text. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.user_operation_color_table` | {py:data}`~claveles.circuit.visualize.DEFAULT_USER_OP_COLOR_TABLE` | Box colour of each {py:mod}`~claveles.circuit.ops.std` operation. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.intrinsic_operation_name_table` | {py:data}`~claveles.circuit.visualize.DEFAULT_INTRINSIC_OP_NAME_TABLE` | The same, for {py:mod}`~claveles.circuit.ops.intrinsic` operations. |
| {py:attr}`~claveles.circuit.visualize.VisualizeConfig.intrinsic_operation_color_table` | {py:data}`~claveles.circuit.visualize.DEFAULT_INTRINSIC_OP_COLOR_TABLE` | Box colour of each intrinsic operation. |

Which pair of tables applies depends on which module the operations came from. A circuit built from
{py:mod}`~claveles.circuit.ops.std` gates, like the sample above, is governed by the `user_operation_*`
tables; the `intrinsic_operation_*` tables take effect for gates taken from
{py:mod}`~claveles.circuit.ops.intrinsic`, or after
{py:meth}`~claveles.circuit.program.CircuitRepr.convert_std_ops_to_intrinsic` has expanded a circuit.

The symbols the two tables give are the ones {ref}`Gates & Operations <sec:gates-symbols>` documents, which is where
to look to read a figure: a {py:mod}`~claveles.circuit.ops.std` operation carries the bare symbol, and an
{py:mod}`~claveles.circuit.ops.intrinsic` operation takes an $\text{intr}$ subscript only where it is a
different function of its arguments than the {py:mod}`~claveles.circuit.ops.std` operation of that name. So a
circuit drawn with {py:class}`intrinsic.BeamSplitter <claveles.circuit.ops.intrinsic.BeamSplitter>` shows
$B_\text{intr}$ while one drawn with {py:class}`std.BeamSplitter <claveles.circuit.ops.std.BeamSplitter>`
shows $B$, and $R$ means the same operation in either.

The examples below vary a few of them. Copy a whole table and change the entries you care about, rather
than passing a partial one.

Larger parameter labels, with {py:attr}`~claveles.circuit.visualize.VisualizeConfig.show_parameters` and
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.parameters_fontsize`:

In [ ]:
make_figure(c, show_parameters=True, parameters_fontsize=15.0);

Red feedforward arrows, with {py:attr}`~claveles.circuit.visualize.VisualizeConfig.feedforward_arrow_style`
and {py:attr}`~claveles.circuit.visualize.VisualizeConfig.feedforward_arrow_color`:

In [ ]:
make_figure(c, feedforward_arrow_style="->", feedforward_arrow_color="red");

Custom mode labels and wider spacing, with
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.qumode_labels` and
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.qumode_hline_spacing`:

In [ ]:
make_figure(c, qumode_labels=["q0", "q1", "q2", "q3"], qumode_hline_spacing=2.0);

A renamed and recoloured operation, wider boxes, and a legend —
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.user_operation_name_table`,
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.user_operation_color_table`,
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.opbox_width`,
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.opbox_vertical_margin` and
{py:attr}`~claveles.circuit.visualize.VisualizeConfig.show_op_legend`:

In [ ]:
from claveles.circuit.visualize import (
    DEFAULT_USER_OP_COLOR_TABLE,
    DEFAULT_USER_OP_NAME_TABLE,
)

user_names = DEFAULT_USER_OP_NAME_TABLE.copy()
user_names["BeamSplitter"] = r"$\mathrm{BS}$"
user_colors = DEFAULT_USER_OP_COLOR_TABLE.copy()
user_colors["PhaseRotation"] = "lightblue"

make_figure(
    c,
    user_operation_name_table=user_names,
    user_operation_color_table=user_colors,
    opbox_vertical_margin=0.5,
    opbox_width=1.2,
    show_op_legend=True,
);